# 01 — Generate Initial Customer Snapshot

## Purpose

This notebook generates a deterministic initial customer snapshot representing data extracted from a CRM source system.

The generated records provide the customer master data required by the Customer 360 model and include CDC metadata for downstream incremental processing.

## Dataset Overview

- 5,000 unique customers
- Multiple countries and regions
- Standard, Premium, and VIP customer segments
- Active and Inactive customer statuses
- Initial CDC operation set to INSERT

## Source System

CRM

## Target Location

`/Volumes/workspace/revenue_leakage_bronze/landing/crm/customers/initial_load`

## 1. Configuration

In [0]:
from pyspark.sql import functions as F


NUM_CUSTOMERS = 5000

LANDING_PATH = (
    "/Volumes/workspace/"
    "revenue_leakage_bronze/"
    "landing"
)

CUSTOMERS_PATH = f"{LANDING_PATH}/crm/customers"
CUSTOMERS_INITIAL_PATH = f"{CUSTOMERS_PATH}/initial_load"

## 2. Define Reference Data

Define reusable reference values for generating deterministic customer names and geographic attributes.

In [0]:
first_names = [
    "Aron", "Sara", "Leon", "Emma", "Noah",
    "Mia", "Liam", "Sofia", "David", "Elena"
]

last_names = [
    "Anderson", "Brown", "Davis", "Garcia", "Johnson",
    "Miller", "Martinez", "Taylor", "Wilson", "Clark"
]

locations = [
    ("United States", "California"),
    ("United States", "Texas"),
    ("United States", "New York"),
    ("United States", "Florida"),
    ("Germany", "Berlin"),
    ("Germany", "Bavaria"),
    ("United Kingdom", "England"),
    ("Canada", "Ontario"),
    ("France", "Ile-de-France"),
    ("Netherlands", "North Holland")
]

first_names_array = F.array(
    *[F.lit(name) for name in first_names]
)

last_names_array = F.array(
    *[F.lit(name) for name in last_names]
)

locations_array = F.array(
    *[
        F.struct(
            F.lit(country).alias("country"),
            F.lit(region).alias("region")
        )
        for country, region in locations
    ]
)

## 3. Generate the Initial Customer Snapshot

Generate unique customer identifiers and enrich each record with deterministic personal, geographic, segmentation, lifecycle, and CDC attributes.

In [0]:
customers_initial_df = (
    spark.range(1, NUM_CUSTOMERS + 1)
    .withColumnRenamed("id", "customer_number")

    .withColumn(
        "customer_id",
        F.format_string("C%06d", F.col("customer_number"))
    )

    .withColumn(
        "first_name",
        F.element_at(
            first_names_array,
            (
                F.pmod(
                    F.col("customer_number") - 1,
                    F.lit(len(first_names))
                ) + 1
            ).cast("int")
        )
    )

    .withColumn(
        "last_name",
        F.element_at(
            last_names_array,
            (
                F.pmod(
                    F.floor(
                        (F.col("customer_number") - 1)
                        / F.lit(len(first_names))
                    ),
                    F.lit(len(last_names))
                ) + 1
            ).cast("int")
        )
    )

    .withColumn(
        "email",
        F.lower(
            F.concat(
                F.col("first_name"),
                F.lit("."),
                F.col("last_name"),
                F.col("customer_number"),
                F.lit("@example.com")
            )
        )
    )

    .withColumn(
        "location",
        F.element_at(
            locations_array,
            (
                F.pmod(
                    F.floor(
                        (F.col("customer_number") - 1) / 100
                    ),
                    F.lit(len(locations))
                ) + 1
            ).cast("int")
        )
    )

    .withColumn("country", F.col("location.country"))
    .withColumn("region", F.col("location.region"))

    .withColumn(
        "segment_bucket",
        F.pmod(
            F.col("customer_number") * 17 + 3,
            F.lit(20)
        )
    )

    .withColumn(
        "customer_segment",
        F.when(F.col("segment_bucket") == 0, "VIP")
        .when(F.col("segment_bucket") <= 5, "Premium")
        .otherwise("Standard")
    )

    .withColumn(
        "status_bucket",
        F.pmod(
            F.col("customer_number") * 13 + 7,
            F.lit(20)
        )
    )

    .withColumn(
        "customer_status",
        F.when(F.col("status_bucket") == 0, "Inactive")
        .otherwise("Active")
    )

    .withColumn(
        "signup_date",
        F.date_add(
            F.lit("2023-01-01").cast("date"),
            F.pmod(
                F.col("customer_number") * 37,
                F.lit(1095)
            ).cast("int")
        )
    )

    .withColumn("operation", F.lit("INSERT"))

    .withColumn(
        "event_timestamp",
        F.col("signup_date").cast("timestamp")
    )

    .select(
        "customer_id",
        "first_name",
        "last_name",
        "email",
        "country",
        "region",
        "customer_segment",
        "signup_date",
        "customer_status",
        "operation",
        "event_timestamp"
    )
)

## 4. Validate the Generated Dataset

Validate row counts, business-key uniqueness, mandatory fields, representative records, and the generated schema before persisting the dataset.

In [0]:
actual_count = customers_initial_df.count()

distinct_customer_count = (
    customers_initial_df
    .select("customer_id")
    .distinct()
    .count()
)

distinct_email_count = (
    customers_initial_df
    .select("email")
    .distinct()
    .count()
)

null_key_count = (
    customers_initial_df
    .filter(
        F.col("customer_id").isNull()
        | F.col("email").isNull()
    )
    .count()
)

assert actual_count == NUM_CUSTOMERS
assert distinct_customer_count == NUM_CUSTOMERS
assert distinct_email_count == NUM_CUSTOMERS
assert null_key_count == 0

print(f"Generated rows: {actual_count:,}")
print(f"Distinct customer IDs: {distinct_customer_count:,}")
print(f"Distinct email addresses: {distinct_email_count:,}")
print(f"Null business keys: {null_key_count:,}")

display(
    customers_initial_df
    .orderBy("customer_id")
    .limit(10)
)

display(
    customers_initial_df
    .groupBy("customer_segment", "customer_status")
    .count()
    .orderBy("customer_segment", "customer_status")
)

customers_initial_df.printSchema()

## 5. Persist the Raw Customer Snapshot

Store the validated customer snapshot as raw JSON files in the CRM landing directory. The initial snapshot is regenerated on every notebook run to keep the synthetic source data reproducible.

In [0]:
# The synthetic initial snapshot is fully regenerated on every run.
(
    customers_initial_df.write
    .format("json")
    .mode("overwrite")
    .save(CUSTOMERS_INITIAL_PATH)
)

saved_customers_df = spark.read.json(
    CUSTOMERS_INITIAL_PATH
)

saved_count = saved_customers_df.count()

assert saved_count == NUM_CUSTOMERS

print(f"Saved rows: {saved_count:,}")
print(f"Target path: {CUSTOMERS_INITIAL_PATH}")

display(
    saved_customers_df
    .orderBy("customer_id")
    .limit(10)
)